# Mesa de Partes Digital–Sismo (Ate) — demostración con 10 casos

**Fundamentos de Programación · Evaluación Parcial**

Este cuaderno contiene los **mismos cinco módulos Python** del proyecto y **10 expedientes ficticios precargados**. Sirve para registrar dos expedientes nuevos, comparar el listado **antes y después** de ordenar, buscar por código y guardar la información.

**Datos ficticios para la sustentación:** no son solicitudes reales ni evidencias de trabajo de campo.

**Cómo usarlo en la universidad:** abre este cuaderno en Colab y selecciona **Entorno de ejecución → Ejecutar todas**. Las celdas crean los cinco archivos y preparan los datos antes de mostrar el menú.

## Preparar la carpeta de esta demostración

Usamos una carpeta separada para **no tocar tus reportes de pruebas anteriores**.

In [ ]:
import os
os.makedirs("/content/Mesa_Partes_Sismo_Demo", exist_ok=True)
%cd /content/Mesa_Partes_Sismo_Demo

## 1. Validaciones

Comprobación de DNI, nombres, distrito, tipos de reporte y otros datos.

In [ ]:
%%writefile validaciones.py
"""Comprobaciones básicas para datos de las solicitudes de Ate."""
from datetime import datetime

DISTRITOS = ["ATE"]

def validar_dni(dni):
    return len(dni) == 8 and dni.isdigit()

def validar_nombre(nombre):
    limpio = nombre.strip()
    if len(limpio) < 3:
        return False
    for letra in limpio:
        if not (letra.isalpha() or letra == " "):
            return False
    return True

def validar_distrito(distrito):
    return distrito.strip().upper() in DISTRITOS

def validar_tipo(tipo):
    return tipo in ("1", "2", "3")

def validar_descripcion(texto):
    return 10 <= len(texto.strip()) <= 150 and "|" not in texto

def validar_personas(texto):
    return texto.isdigit() and 0 <= int(texto) <= 500

def validar_si_no(texto):
    return texto.strip().upper() in ("S", "N")

def validar_fecha(fecha):
    try:
        datetime.strptime(fecha, "%d/%m/%Y")
        return True
    except ValueError:
        return False


## 2. Clasificación

Asignación de puntajes y niveles CRITICO, ALTO, MEDIO y BAJO.

In [ ]:
%%writefile clasificacion.py
"""Clasificación de expedientes en cuatro niveles."""
PALABRAS_GRAVES = ["atrapado", "atrapada", "derrumbe", "colapso", "incendio",
                    "fuga de gas", "herido", "sin respirar", "desaparecido"]
RANGO = {"CRITICO": 1, "ALTO": 2, "MEDIO": 3, "BAJO": 4}

def contiene_palabra_grave(descripcion):
    texto = descripcion.lower()
    for palabra in PALABRAS_GRAVES:
        if palabra in texto:
            return True
    return False

def calcular_puntaje(tipo, personas, heridos, descripcion):
    puntaje = 0
    if tipo == "2":
        puntaje += 3
    elif tipo == "1":
        puntaje += 2
    else:
        puntaje += 1
    if personas >= 10:
        puntaje += 3
    elif personas >= 4:
        puntaje += 2
    elif personas >= 1:
        puntaje += 1
    if heridos == "S":
        puntaje += 3
    if contiene_palabra_grave(descripcion):
        puntaje += 3
    return puntaje

def clasificar(puntaje):
    if puntaje >= 8:
        return "CRITICO"
    if puntaje >= 5:
        return "ALTO"
    if puntaje >= 3:
        return "MEDIO"
    return "BAJO"


## 3. Expedientes

Registro, búsqueda secuencial y ordenamiento burbuja.

In [ ]:
%%writefile expedientes.py
"""Operaciones de registro, consulta y ordenamiento."""
import clasificacion as cla

TIPOS = {"1": "Daño", "2": "Ayuda", "3": "Atención"}

def generar_codigo(codigos, anio):
    return f"SIS-{anio}-{len(codigos) + 1:04d}"

def registrar(datos, dni, nombre, distrito, tipo, descripcion, personas, heridos, fecha, anio):
    codigo = generar_codigo(datos["codigos"], anio)
    puntaje = cla.calcular_puntaje(tipo, int(personas), heridos.upper(), descripcion)
    prioridad = cla.clasificar(puntaje)
    datos["codigos"].append(codigo)
    datos["dnis"].append(dni)
    datos["nombres"].append(nombre.strip().title())
    datos["distritos"].append(distrito.strip().upper())
    datos["tipos"].append(TIPOS[tipo])
    datos["descripciones"].append(descripcion.strip())
    datos["personas"].append(str(int(personas)))
    datos["heridos"].append(heridos.upper())
    datos["fechas"].append(fecha)
    datos["prioridades"].append(prioridad)
    return codigo, prioridad

def buscar_por_codigo(codigos, codigo):
    for i in range(len(codigos)):
        if codigos[i] == codigo.strip().upper():
            return i
    return -1

def _burbuja(claves, datos):
    n = len(claves)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if claves[j] > claves[j+1]:
                claves[j], claves[j+1] = claves[j+1], claves[j]
                for campo in datos:
                    arreglo = datos[campo]
                    arreglo[j], arreglo[j+1] = arreglo[j+1], arreglo[j]

def ordenar_por_codigo(datos):
    _burbuja(list(datos["codigos"]), datos)

def ordenar_por_prioridad(datos):
    ordenar_por_codigo(datos)
    claves = []
    for prioridad in datos["prioridades"]:
        claves.append(cla.RANGO[prioridad])
    _burbuja(claves, datos)

def contar_por_prioridad(prioridades):
    conteo = {"CRITICO": 0, "ALTO": 0, "MEDIO": 0, "BAJO": 0}
    for p in prioridades:
        conteo[p] += 1
    return conteo


## 4. Archivos

Lectura y escritura del archivo TXT con expedientes.

In [ ]:
%%writefile archivos.py
"""Guardar y recuperar expedientes desde un archivo de texto."""
import os

RUTA = "reportes_sismo.txt"
CAMPOS = ["codigos", "dnis", "nombres", "distritos", "tipos",
          "descripciones", "personas", "heridos", "fechas", "prioridades"]

def nuevo_almacen():
    datos = {}
    for campo in CAMPOS:
        datos[campo] = []
    return datos

def guardar(datos, ruta=RUTA):
    with open(ruta, "w", encoding="utf-8") as archivo:
        for i in range(len(datos["codigos"])):
            campos_reporte = []
            for campo in CAMPOS:
                campos_reporte.append(datos[campo][i])
            archivo.write("|".join(campos_reporte) + "\n")

def cargar(ruta=RUTA):
    datos = nuevo_almacen()
    if not os.path.exists(ruta):
        return datos
    with open(ruta, "r", encoding="utf-8") as archivo:
        for linea in archivo:
            partes = linea.rstrip("\n").split("|")
            if len(partes) == len(CAMPOS):
                for i in range(len(CAMPOS)):
                    datos[CAMPOS[i]].append(partes[i])
    return datos


## 5. Menú principal

Seis opciones para interactuar con el programa.

In [ ]:
%%writefile main.py
"""Mesa de Partes Digital-Sismo, caso Ate."""
from datetime import datetime
import validaciones as val
import expedientes as exp
import archivos as arc

def pedir_dato(mensaje, tipo, error):
    """Pide un dato hasta que pase la validación correspondiente."""
    while True:
        dato = input(mensaje).strip()
        correcto = False

        if tipo == "dni":
            correcto = val.validar_dni(dato)
        elif tipo == "nombre":
            correcto = val.validar_nombre(dato)
        elif tipo == "distrito":
            correcto = val.validar_distrito(dato)
        elif tipo == "reporte":
            correcto = val.validar_tipo(dato)
        elif tipo == "descripcion":
            correcto = val.validar_descripcion(dato)
        elif tipo == "personas":
            correcto = val.validar_personas(dato)
        elif tipo == "heridos":
            correcto = val.validar_si_no(dato)
        elif tipo == "fecha":
            correcto = val.validar_fecha(dato)

        if correcto:
            return dato
        print(" [!] " + error)

def mostrar(datos, i):
    print(f" [{datos['prioridades'][i]}] {datos['codigos'][i]} | "
          f"{datos['nombres'][i]} | {datos['distritos'][i]} | "
          f"{datos['tipos'][i]} | {datos['personas'][i]} personas | "
          f"{datos['descripciones'][i]}")

def menu():
    datos = arc.cargar()
    anio = datetime.now().year
    opcion = ""
    while opcion != "6":
        print("\n=== MESA DE PARTES DIGITAL-SISMO (ATE) ===")
        print("1. Registrar     2. Buscar por código     3. Ordenar por prioridad")
        print("4. Listar       5. Resumen               6. Salir")
        opcion = input("Opción: ").strip()
        if opcion == "1":
            dni = pedir_dato("DNI (8 dígitos): ", "dni", "DNI inválido.")
            nombre = pedir_dato("Nombre completo: ", "nombre", "Nombre inválido.")
            distrito = pedir_dato("Distrito: ", "distrito", "Por ahora solo se registra ATE.")
            tipo = pedir_dato("Tipo (1 Daño, 2 Ayuda, 3 Atención): ", "reporte", "Elija 1, 2 o 3.")
            desc = pedir_dato("Descripción (10-150 caracteres): ", "descripcion", "Descripción inválida.")
            personas = pedir_dato("Personas afectadas: ", "personas", "Número de 0 a 500.")
            heridos = pedir_dato("¿Hay heridos? (S/N): ", "heridos", "Responda S o N.")
            fecha = pedir_dato("Fecha (dd/mm/aaaa): ", "fecha", "Fecha inválida.")
            codigo, prioridad = exp.registrar(datos, dni, nombre, distrito, tipo, desc, personas, heridos, fecha, anio)
            arc.guardar(datos)
            print(f" Registrado: {codigo} | Prioridad: {prioridad}")
        elif opcion == "2":
            i = exp.buscar_por_codigo(datos["codigos"], input("Código: "))
            if i == -1:
                print(" Expediente no encontrado.")
            else:
                mostrar(datos, i)
        elif opcion == "3":
            exp.ordenar_por_prioridad(datos)
            arc.guardar(datos)
            print(" Ordenado: casos críticos primero.")
        elif opcion == "4":
            if not datos["codigos"]:
                print(" No hay expedientes.")
            for i in range(len(datos["codigos"])):
                mostrar(datos, i)
        elif opcion == "5":
            conteo = exp.contar_por_prioridad(datos["prioridades"])
            for nivel in conteo:
                print(f" {nivel}: {conteo[nivel]}")
        elif opcion != "6":
            print(" Opción no válida.")

if __name__ == "__main__":
    menu()


## 6. Diez expedientes de demostración (ficticios)

El archivo siguiente contiene 10 expedientes con prioridades mezcladas: **3 CRITICO, 2 ALTO, 3 MEDIO y 2 BAJO**. Los códigos van de `SIS-2026-0001` a `SIS-2026-0010`. Se presentan deliberadamente sin ordenar para demostrar la opción 3.

In [ ]:
%%writefile casos_demo_sismo_10.txt
SIS-2026-0001|00000001|Ana Prueba|ATE|Atención|Consulta sobre el punto de informacion vecinal|0|N|21/10/2026|BAJO
SIS-2026-0002|00000002|Luis Ejemplo|ATE|Daño|Grietas en una pared de la vivienda|1|N|21/10/2026|MEDIO
SIS-2026-0003|00000003|Maria Simulada|ATE|Ayuda|Familias atrapadas en edificio con colapso|12|S|21/10/2026|CRITICO
SIS-2026-0004|00000004|Carlos Ensayo|ATE|Ayuda|Se requieren agua y mantas para cinco vecinos|5|N|21/10/2026|ALTO
SIS-2026-0005|00000005|Rosa Ficticia|ATE|Atención|Pregunta por horario de atencion municipal|0|N|21/10/2026|BAJO
SIS-2026-0006|00000006|Pedro Prueba|ATE|Daño|Derrumbe parcial cerca de viviendas afectadas|9|S|21/10/2026|CRITICO
SIS-2026-0007|00000007|Elena Ejemplo|ATE|Atención|Consultan por la ubicacion del punto de apoyo|4|N|21/10/2026|MEDIO
SIS-2026-0008|00000008|Jorge Simulado|ATE|Daño|Viviendas con grietas y otros daños materiales|11|N|21/10/2026|ALTO
SIS-2026-0009|00000009|Lucia Ensayo|ATE|Ayuda|Solicitan informacion sobre reparto de alimentos|0|N|21/10/2026|MEDIO
SIS-2026-0010|00000010|Diana Ficticia|ATE|Ayuda|Vecino herido necesita traslado inmediato|2|S|21/10/2026|CRITICO


## 7. Cargar la demostración una sola vez

Si `reportes_sismo.txt` no existe en la carpeta de demostración, se crea a partir de los diez casos. Si ya existe, **no se sobrescribe**: de esta manera puedes agregar dos y conservar los 12 expedientes al ejecutar de nuevo en la misma sesión.

In [ ]:
import os
import shutil

if not os.path.exists("reportes_sismo.txt"):
    shutil.copyfile("casos_demo_sismo_10.txt", "reportes_sismo.txt")
    print("Demostración preparada con 10 expedientes ficticios.")
else:
    print("Se conserva el archivo de trabajo de esta sesión.")

with open("reportes_sismo.txt", "r", encoding="utf-8") as archivo:
    cantidad = len(archivo.readlines())
print("Expedientes disponibles:", cantidad)


## 8. Ejecutar el programa

En el menú, primero elige **4 (Listar)**: verás los 10 expedientes mezclados. Luego elige **1 (Registrar)** dos veces: deben aparecer `SIS-2026-0011` y `SIS-2026-0012`. Después elige **4 (Listar)** para ver los 12 sin ordenar, **3 (Ordenar por prioridad)** y **4 (Listar)** otra vez. Para terminar, prueba **2 (Buscar por código)**, **5 (Resumen)** y **6 (Salir)**.

**Los registros se guardan automáticamente** al registrar y al ordenar; no necesitas agregar una opción nueva al menú.

In [ ]:
%run main.py

## 9. Descargar los 12 expedientes y conservar la evidencia (opcional)

Al terminar la demostración, elige **6. Salir**. Después, ejecuta la siguiente celda para descargar el TXT. Recuerda que Colab puede eliminar los archivos de la sesión al desconectarse; el **cuaderno sí permanece guardado** en tu cuenta.

In [ ]:
from google.colab import files
files.download("reportes_sismo.txt")

### ¿Quieres volver a empezar con exactamente 10 casos?

Después de salir del menú, crea **una celda nueva** y ejecuta estas dos líneas **solo si quieres descartar los cambios de la demostración**:

```python
import shutil
shutil.copyfile("casos_demo_sismo_10.txt", "reportes_sismo.txt")
```

Este reinicio es manual para que **«Ejecutar todas» no borre tus 12 casos** cuando continúas trabajando en la misma sesión.